## はじめに

このノートブックでは、GlacierStyle ECサイトのデータを Snowflake に取り込み、**Bronze 層**と **Silver 層**のテーブルを作成します。音声や PDF のような非構造化データも、ステージに置けば SQL でテーブルにできることを体験します。

> **メダリオンアーキテクチャ**: データを加工の段階ごとに3つの層に分けて管理します。
> - **Bronze 層**: 取り込んだままの生データ（`RAW_` で始まるテーブル）
> - **Silver 層**: 整えて分析に使える形にしたデータ（`DIM_`・`FACT_`、Part 2 の `CALL_ANALYSIS`・`DOC_ATTRIBUTES`）
> - **Gold 層**: 検索や AI アシスタントにそのまま使う形にしたデータ（Part 2 の `KNOWLEDGE_CHUNKS`）

**前提条件:**
- `setup_20261216.sql` が実行済みであること（データベース・スキーマ・ステージ・GitHub 連携の設定と、データファイルのコピーが完了）

**主な処理内容:**
1. 構造化データ（CSV）のロード
2. 音声ファイルの文字起こし（AI_TRANSCRIBE）
3. PDF ドキュメントの解析（AI_PARSE_DOCUMENT）

**生成されるテーブル:**

*Silver 層: ディメンションテーブル（マスタデータ）*
- `DIM_CUSTOMERS`: 顧客マスタ（300件）
- `DIM_PRODUCTS`: 商品マスタ（55件）

*Silver 層: ファクトテーブル（トランザクションデータ）*
- `FACT_ORDERS`: EC取引データ（2,913件。入荷ロット番号付き）

*Bronze 層: 非構造化データテーブル*
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みのコールセンター通話（88件）
- `RAW_VOICE_LOGS`: 通話音声の文字起こし結果（10件）
- `RAW_DOCUMENTS`: PDF ドキュメントのページ単位のテキスト（11ファイル・99ページ）

> **料金について**: AI 関数は処理量に応じてクレジットを消費します。実行前に[最新の料金](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-cost)を確認してください。このノートブックの AI 処理は、音声10件と PDF 99ページです。

> **途中で遅れた場合**: `checkpoint_20261216.sql` の「Part 1」を実行すると、このノートブックの完了時点のテーブルを復元できます。

In [ ]:
-- ============================================================================
-- 環境設定
-- ============================================================================
-- 使用するロール、ウェアハウス、スキーマを設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE SCHEMA GLACIERSTYLE_DB.EC_ANALYTICS_SCHEMA;

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

images_dir = 'images/part1/'

def display_image(image_file: str) -> None:
    image_path = os.path.join(images_dir, image_file)
    img = Image.open(image_path)
    plt.figure(figsize=(15, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

## 1. 構造化データのロード

ステージにコピーした CSV を、`COPY INTO` でテーブルに読み込みます。CSV の1行目（ヘッダー）の列名とテーブルの列名を対応させて読み込むため、列の順番を気にする必要はありません。

`FACT_ORDERS` の `lot_no` は商品の入荷ロットです。Part 2 で、通話の内容とこの列を組み合わせて分析します。

In [ ]:
-- ============================================================================
-- テーブル定義の作成と CSV のロード
-- ============================================================================
-- CSV のヘッダー行を列名として使うファイル形式
CREATE OR REPLACE FILE FORMAT HANDSON_CSV
    TYPE = CSV
    PARSE_HEADER = TRUE
    FIELD_OPTIONALLY_ENCLOSED_BY = '"';

-- 商品マスタ
CREATE OR REPLACE TABLE DIM_PRODUCTS (
    product_id VARCHAR, product_name VARCHAR,
    category_l1 VARCHAR, category_l2 VARCHAR, category_l3 VARCHAR,
    brand VARCHAR, list_price NUMBER, cost_price NUMBER, launch_date DATE
);

-- 顧客マスタ
CREATE OR REPLACE TABLE DIM_CUSTOMERS (
    customer_id VARCHAR, last_name VARCHAR, first_name VARCHAR, gender VARCHAR,
    birth_date DATE, email VARCHAR, phone VARCHAR,
    prefecture VARCHAR, city VARCHAR, registration_date DATE, membership_tier VARCHAR
);

-- 注文データ（lot_no は商品の入荷ロット）
CREATE OR REPLACE TABLE FACT_ORDERS (
    order_id VARCHAR, order_datetime TIMESTAMP_NTZ, customer_id VARCHAR, product_id VARCHAR,
    quantity NUMBER, unit_price NUMBER, discount_amount NUMBER, total_amount NUMBER,
    order_channel VARCHAR, order_status VARCHAR, lot_no VARCHAR
);

-- 文字起こし済みの通話
CREATE OR REPLACE TABLE RAW_CALL_TRANSCRIPTS (
    call_id VARCHAR, call_datetime TIMESTAMP_NTZ, customer_id VARCHAR, order_id VARCHAR,
    product_id VARCHAR, agent_id VARCHAR, call_duration_sec NUMBER, transcript_text VARCHAR
);

-- CSV の列名とテーブルの列名を対応させてロード
COPY INTO DIM_PRODUCTS FROM @DATA_STAGE FILES = ('products.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO DIM_CUSTOMERS FROM @DATA_STAGE FILES = ('customers.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO FACT_ORDERS FROM @DATA_STAGE FILES = ('orders.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;
COPY INTO RAW_CALL_TRANSCRIPTS FROM @DATA_STAGE FILES = ('call_transcripts.csv')
    FILE_FORMAT = (FORMAT_NAME = HANDSON_CSV) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

In [ ]:
-- ============================================================================
-- ロード結果の確認（文字起こし済みの通話を3件表示）
-- ============================================================================
SELECT call_id, call_datetime, product_id, LEFT(transcript_text, 80) AS transcript_preview
FROM RAW_CALL_TRANSCRIPTS
ORDER BY call_id
LIMIT 3;

## 2. 音声ファイルの文字起こし

コールセンターの通話音声（MP3、10件）を `AI_TRANSCRIBE` で文字起こしします。ファイルは `TO_FILE('@ステージ', 'パス')` で指定します。

この10件は文字起こしを体験するためのサンプルで、注文や商品のデータとはつながっていません。結果は `RAW_VOICE_LOGS` に保存し、Part 2 で文字起こし済みの88件（`RAW_CALL_TRANSCRIPTS`）とあわせて分析します。

In [ ]:
display_image('ai_transcribe.png')

In [ ]:
-- ============================================================================
-- AI_TRANSCRIBE: 1件だけ文字起こしを試してみよう
-- ============================================================================
-- 結果は JSON で返り、text に文字起こし、audio_duration に音声の長さ（秒）が入ります
SELECT AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', 'voice_logs/CALL_001.mp3')) AS transcribe_result;

In [ ]:
-- ============================================================================
-- 音声10件を文字起こしし、RAW_VOICE_LOGS に保存
-- ============================================================================
-- DIRECTORY(@DATA_STAGE) でステージ上のファイル一覧を取得し、MP3 だけを処理します
CREATE OR REPLACE TABLE RAW_VOICE_LOGS AS
WITH transcribed AS (
    SELECT
        SPLIT_PART(relative_path, '/', -1) AS audio_file,
        AI_TRANSCRIBE(TO_FILE('@DATA_STAGE', relative_path)) AS result
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE 'voice_logs/%.mp3'
)
SELECT
    'VOICE-' || REPLACE(audio_file, '.mp3', '') AS call_id,
    audio_file,
    result:audio_duration::NUMBER(10, 1) AS audio_duration_sec,
    result:text::VARCHAR AS transcript_text
FROM transcribed;

-- 文字起こし結果を確認
SELECT call_id, audio_duration_sec, LEFT(transcript_text, 80) AS transcript_preview
FROM RAW_VOICE_LOGS
ORDER BY call_id;

## 3. PDF ドキュメントの解析

FAQ、運用マニュアル、品質報告書、通達、店舗報告などの PDF（11ファイル）を `AI_PARSE_DOCUMENT` で解析します。

- `'mode': 'LAYOUT'`: 見出しや表の構造を保ったまま、Markdown 形式のテキストにします
- `'page_split': true`: ページごとに結果を分けます。後で「どのページに書いてあるか」を残せます

In [ ]:
display_image('ai_parse_document.png')

In [ ]:
-- ============================================================================
-- AI_PARSE_DOCUMENT: 品質報告書（1ページ）の解析を試してみよう
-- ============================================================================
-- pages の各要素の content に、ページの本文が Markdown で入ります
SELECT AI_PARSE_DOCUMENT(
    TO_FILE('@DATA_STAGE', 'documents/quality_report_202611.pdf'),
    {'mode': 'LAYOUT', 'page_split': true}
) AS parse_result;

In [ ]:
-- ============================================================================
-- PDF 11ファイルをページ単位でテーブルにする
-- ============================================================================
CREATE OR REPLACE TABLE RAW_DOCUMENTS AS
WITH parsed AS (
    SELECT
        SPLIT_PART(relative_path, '/', -1) AS file_name,
        AI_PARSE_DOCUMENT(
            TO_FILE('@DATA_STAGE', relative_path),
            {'mode': 'LAYOUT', 'page_split': true}
        ) AS doc
    FROM DIRECTORY(@DATA_STAGE)
    WHERE relative_path ILIKE '%.pdf'
)
-- pages の配列を FLATTEN で1ページ1行に展開
SELECT
    p.file_name,
    pg.index + 1 AS page_no,
    pg.value:content::VARCHAR AS page_text
FROM parsed p,
LATERAL FLATTEN(input => p.doc:pages) pg;

-- ファイルごとのページ数と、1ページ目の冒頭を確認
SELECT
    file_name,
    COUNT(*) AS page_count,
    LEFT(MAX(IFF(page_no = 1, page_text, NULL)), 60) AS first_page_preview
FROM RAW_DOCUMENTS
GROUP BY file_name
ORDER BY file_name;

## 4. データ確認

作成した Bronze 層と Silver 層のテーブルの件数を確認します。

In [ ]:
-- ============================================================================
-- 各テーブルのレコード数を確認
-- ============================================================================
SELECT 'DIM_PRODUCTS' AS table_name, COUNT(*) AS record_count, '商品マスタ（55件）' AS description FROM DIM_PRODUCTS
UNION ALL
SELECT 'DIM_CUSTOMERS', COUNT(*), '顧客マスタ（300件）' FROM DIM_CUSTOMERS
UNION ALL
SELECT 'FACT_ORDERS', COUNT(*), '注文データ（2,913件）' FROM FACT_ORDERS
UNION ALL
SELECT 'RAW_CALL_TRANSCRIPTS', COUNT(*), '文字起こし済みの通話（88件）' FROM RAW_CALL_TRANSCRIPTS
UNION ALL
SELECT 'RAW_VOICE_LOGS', COUNT(*), '通話音声の文字起こし（10件）' FROM RAW_VOICE_LOGS
UNION ALL
SELECT 'RAW_DOCUMENTS', COUNT(*), 'PDF のページ（99ページ）' FROM RAW_DOCUMENTS;

## まとめ

このノートブックでは、構造化データと非構造化データを取り込み、Bronze 層と Silver 層のテーブルを作成しました。

### 作成したテーブル一覧

**Silver 層: ディメンションテーブル（マスタデータ）**
- `DIM_CUSTOMERS`: 顧客マスタ
- `DIM_PRODUCTS`: 商品マスタ

**Silver 層: ファクトテーブル（トランザクションデータ）**
- `FACT_ORDERS`: EC取引データ

**Bronze 層: 非構造化データテーブル**
- `RAW_CALL_TRANSCRIPTS`: 文字起こし済みの通話
- `RAW_VOICE_LOGS`: 通話音声（文字起こし済み）
- `RAW_DOCUMENTS`: PDF ドキュメント（ページ単位で解析済み）

### 使用した Cortex AI 関数

- `AI_TRANSCRIBE`: 音声ファイルの文字起こし
- `AI_PARSE_DOCUMENT`: PDF ドキュメントの解析

### 次のステップ

- **Part 2**: Bronze 層のテキストから、AI 関数で問い合わせ種別・感情・解約リスクなどを取り出し、Silver 層と Gold 層のテーブルを作成します